[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wildertrek/catcher-in-the-cache/blob/main/notebooks/04_synthetic_characters.ipynb)

# 04: The Synthetic Character Substrate (data card)

The 20 synthetic characters are the out-of-cache probe at the heart of the paper:
original characters absent from any training corpus, used to separate *retrieval*
(recognizing a known character) from *measurement* (reading traits from text). This
notebook documents what they are, how they were generated, and the files they produced.

## Setup

**In Colab, run the cell below first.** It clones the companion repository and installs dependencies (~30 s), then changes into `notebooks/` so the relative paths in this notebook resolve. Run locally, the cell is a no-op.

In [1]:
# Colab setup: clone the companion repo + install dependencies (no-op when run locally).
import sys, os, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("catcher-in-the-cache"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Wildertrek/catcher-in-the-cache.git"], check=True)
    os.chdir("catcher-in-the-cache/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())

## How they were generated

Provenance: the cross-rater panel, pre-registration A3.3b Upgrade 2 (synthetic-character substrate to
decouple priors from evidence). Script: the generation scripts in the umbrella repository.

1. **Generation, Claude Opus 4.6 (temperature 0.8).** One structured prompt asked for 20
   genuinely novel fictional characters absent from any known literature, film, TV, comics,
   or games, with a deliberate trait-space requirement: include **high-H / low-A_HEX**
   (sincere but irritable), **low-H / high-A_HEX** (manipulative but patient), and canonically
   aligned profiles. For each: a ~450-word evidence pack (60% dialogue across varied contexts,
   40% narrator commentary describing observable behavior, with NO trait labels), plus target
   OCEAN and HEXACO scores on [-1, +1].
2. **Novelty negative-control.** Each evidence pack was shown to three other LLMs (Claude
   Sonnet 4.5, GPT-4o, Gemini 2.5 Pro) asked to identify the source; any character a model
   could place was flagged for regeneration. This is what makes them out-of-corpus.

**Honest caveats (load-bearing).** The characters, the evidence-pack text, AND the target
scores are all LLM-generated, under a human-designed decoupling constraint. So both the inputs
(text) and the labels (targets) are LLM-produced. The substrate decouples priors from evidence
by being out-of-corpus, but the gold-standard validation remains a human-rater panel, and
whether the decoupled signal is genuinely encoded in the text (vs only in the target) is exactly
the open question the cache-free regressor work probes.

## From v0 (LLM-set labels) to v1 (human-curated labels)

The data card above is the **v0** substrate: text *and* targets are LLM-produced. That is fine for
the coverage argument (where a character sits in trait space is verified independently by embeddings),
but it is circular for the cache-free regressor, whose labels must not come from the same mechanism it
is meant to be independent of. The fix is to keep the LLM *prose* and replace the LLM *targets* with
human judgment. That happens in the curation workspace, `synthetic_characters_curation.md`: a reader
adjusts the author-locked targets (a rating task, not a writing task), and those become the cache-free
labels. The gold standard remains a multi-rater human panel; one curator is a valid pilot toward it.

In [2]:
# v0 -> v1: the curation path that swaps LLM-set labels for human judgment
import re
from pathlib import Path
cur = Path('paper_artifacts/pivot6_hexaco_atlas/synthetic_characters_curation.md')
if not cur.exists(): cur = Path('../paper_artifacts/pivot6_hexaco_atlas/synthetic_characters_curation.md')
txt = cur.read_text()
entries = re.split(r'\n## \d+\. ', txt)[1:]
curated = sum(1 for b in entries if '(none yet)' not in b.split('### Author notes')[-1])
v1 = cur.parent / 'synthetic_characters_substrate_v1.json'
print(f"curation workspace : {cur.name}")
print(f"  entries staged for human review  : {len(entries)}")
print(f"  entries with author notes filled : {curated}")
print(f"  v1 substrate (human-locked labels): {'present' if v1.exists() else 'not yet built (still v0, LLM-set targets)'}")
print("\nEdit the author-locked targets in the .md; the curated values become the cache-free labels.")

curation workspace : synthetic_characters_curation.md
  entries staged for human review  : 20
  entries with author notes filled : 0
  v1 substrate (human-locked labels): not yet built (still v0, LLM-set targets)

Edit the author-locked targets in the .md; the curated values become the cache-free labels.


In [3]:
%pip install -q pandas numpy plotly
import json, numpy as np, pandas as pd
from pathlib import Path
p=Path('paper_artifacts/pivot6_hexaco_atlas/synthetic_characters_substrate_v0.json')
if not p.exists(): p=Path('../paper_artifacts/pivot6_hexaco_atlas/synthetic_characters_substrate_v0.json')
D=json.loads(p.read_text()); chars=D['characters']
H=np.array([c['target_HEXACO']['H'] for c in chars]); A=np.array([c['target_HEXACO']['A_HEX'] for c in chars])
print(f"{len(chars)} characters, generated {D.get('generated_at')}")
print(f"designed r(H, A_HEX) across the 20 = {np.corrcoef(H,A)[0,1]:+.3f}  (deliberately decoupled)")



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/jsr/Documents/GitHub/mindbench/mindbench-env/bin/python3.12 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


20 characters, generated 2026-05-17T16:32:22.657664+00:00
designed r(H, A_HEX) across the 20 = -0.744  (deliberately decoupled)


## The 20 characters at a glance

Each carries an author-constraint design archetype: *sincere-but-harsh* (high H, low A_HEX),
*charming-but-crooked* (low H, high A_HEX), or *aligned* (H and A_HEX same sign).

In [4]:
def archetype(h,a):
    if h>0.3 and a<-0.3: return 'sincere-but-harsh (H+ A_HEX-)'
    if h<-0.3 and a>0.3: return 'charming-but-crooked (H- A_HEX+)'
    return 'aligned / other'
rows=[]
for c in chars:
    o=c['target_OCEAN']; h=c['target_HEXACO']
    rows.append({'name':c['name'],'work':c['work_title'],
        'O':o['O'],'C':o['C'],'E':o['E'],'A':o['A'],'N':o['N'],
        'H':h['H'],'A_HEX':h['A_HEX'],'design':archetype(h['H'],h['A_HEX'])})
df=pd.DataFrame(rows)
print('design archetype counts:'); print(df['design'].value_counts().to_string())
df


design archetype counts:
design
aligned / other                     9
sincere-but-harsh (H+ A_HEX-)       6
charming-but-crooked (H- A_HEX+)    5


,name,work,O,C,E,A,N,H,A_HEX,design
0,Dorrit Klaes,The Lichfield Dispatches,0.10,0.90,-0.20,-0.60,-0.10,0.85,-0.70,sincere-but-harsh (H+ A_HEX-)
1,Tessaro Vinn,A Gilded Residue,0.30,0.60,0.70,0.50,-0.60,-0.80,0.80,charming-but-crooked (H- A_HEX+)
2,Pella Sond,Undertow Season,0.80,-0.60,0.20,0.80,0.85,0.50,0.70,aligned / other
3,Gren Talvic,Iron and Almanac,-0.50,0.95,-0.70,-0.40,-0.50,0.60,-0.50,sincere-but-harsh (H+ A_HEX-)
4,Ysmé Charoud,The Perfumer's Balcony,0.95,-0.30,0.90,0.60,-0.20,0.40,0.50,aligned / other
5,Dorrit Vashem,The Glassblower's Accord,-0.10,0.70,0.10,-0.60,0.30,0.80,-0.70,sincere-but-harsh (H+ A_HEX-)
6,Caelum Trist,Pale Meridian,0.30,0.60,0.50,0.20,-0.70,-0.80,0.80,charming-but-crooked (H- A_HEX+)
7,Fenra Quillen,The Understory Cartographs,0.80,0.80,0.20,0.60,-0.30,0.60,0.60,aligned / other
8,Marek Solenne,Ironweed Summer,0.00,0.70,-0.20,-0.70,0.20,0.90,-0.80,sincere-but-harsh (H+ A_HEX-)
9,Lirien Dace,The Sable Consort,0.40,0.50,0.60,0.40,-0.60,-0.90,0.90,charming-but-crooked (H- A_HEX+)


## The decoupling, visualized

Canonical literature clusters on the fused diagonal (H and A_HEX rise together). These
synthetics were generated to populate the off-diagonal corners that canonical characters
avoid, which is what makes them a fair out-of-cache test.

In [5]:
import plotly.graph_objects as go
col={'sincere-but-harsh (H+ A_HEX-)':'#0984E3','charming-but-crooked (H- A_HEX+)':'#D63031','aligned / other':'#00b894'}
fig=go.Figure()
for d,c in col.items():
    sub=df[df['design']==d]
    fig.add_trace(go.Scatter(x=sub['H'],y=sub['A_HEX'],mode='markers+text',name=d,
        marker=dict(size=12,color=c),text=sub['name'],textposition='top center',textfont=dict(size=8)))
fig.add_hline(y=0,line_color='#dfe6e9'); fig.add_vline(x=0,line_color='#dfe6e9')
fig.update_layout(template='plotly_white',height=560,title='Designed H vs A_HEX targets (r = -0.74): the decoupled corners',
    xaxis_title='Honesty-Humility (H)',yaxis_title='HEXACO-Agreeableness (A_HEX)',
    xaxis_range=[-1,1],yaxis_range=[-1,1],legend=dict(orientation='h',y=-0.18))
fig.show()


## A second decoupling axis: which corners are worth filling?

The 20 characters above all vary on one plane, **H x A_HEX** (the communion bipolarity). But canonical
literature fusions more than one trait pair. Averaged across the 25 panel raters, the strongest fusions are:

| Trait pair | Canonical mean r (25-rater panel) | Role |
|---|---|---|
| **H ↔ A_HEX** | **+0.75** | axis 1, saturated by the 20 chars |
| O ↔ A | +0.32 | strongest OCEAN fusion, but A overlaps communion |
| **C ↔ N** | **−0.29** | strongest fusion orthogonal to communion → **axis 2** |
| E ↔ N | −0.26 | candidate axis 3 |

Opening a second, orthogonal axis turns "literature fusions communion" into "literature fusions *several*
trait pairs," and gives the cache-free regressor a harder, non-redundant test. **C x N** is the natural
choice: fiction pairs discipline with calm and anxiety with chaos, so the high-C/high-N corner (the
disciplined-but-anxious type) is nearly empty. The curation file carries a **worked example**, *Edda Sarn*,
showing how to author a character into this corner; it is a copy-this template, not a labeled corpus member.

In [6]:
# Second axis C x N: reconstruct canonical per-char OCEAN by averaging the 25 panel raters
# (the duplicate Grok 4.3 run, xai_2, is excluded so this is the standardized 25-rater panel)
import glob, collections, statistics as st
base = Path('paper_artifacts/pivot6_hexaco_atlas')
if not base.exists(): base = Path('../paper_artifacts/pivot6_hexaco_atlas')
acc = collections.defaultdict(lambda: collections.defaultdict(list))
for f in glob.glob(str(base/'ocean_ratings_*.json')):
    if 'synth' in f or 'xai_2' in f: continue
    for p in json.load(open(f)).get('predictions', []):
        pr = p.get('parsed') or {}; cid = p.get('coref_id')
        for t in ('C','N'):
            if isinstance(pr.get(t),(int,float)): acc[cid][t].append(pr[t])
cC=[st.mean(tv['C']) for tv in acc.values() if tv.get('C') and tv.get('N')]
cN=[st.mean(tv['N']) for tv in acc.values() if tv.get('C') and tv.get('N')]
rcn=float(np.corrcoef(cC,cN)[0,1])
sC=[c['target_OCEAN']['C'] for c in chars]; sN=[c['target_OCEAN']['N'] for c in chars]
in_hi=sum(1 for c,n in zip(sC,sN) if c>0.3 and n>0.3)

import plotly.graph_objects as go
fig=go.Figure()
for (x0,x1,y0,y1) in [(0.3,1,0.3,1),(-1,-0.3,-1,-0.3)]:
    fig.add_shape(type='rect',x0=x0,x1=x1,y0=y0,y1=y1,fillcolor='#a29bfe',opacity=0.30,line_width=0,layer='below')
fig.add_trace(go.Scatter(x=cC,y=cN,mode='markers',name=f'canonical (n={len(cC)}, r={rcn:+.2f})',marker=dict(size=6,color='#b2bec3')))
fig.add_trace(go.Scatter(x=sC,y=sN,mode='markers',name='synthetic (20, H/A_HEX-designed)',marker=dict(size=9,color='#D63031')))
fig.add_hline(y=0,line_color='#dfe6e9'); fig.add_vline(x=0,line_color='#dfe6e9')
fig.update_layout(template='plotly_white',height=520,xaxis_range=[-1,1],yaxis_range=[-1,1],
    title='Second axis C x N: canonical anti-couples them; the high-C/high-N corner (shaded) is the gap',
    xaxis_title='Conscientiousness (C)',yaxis_title='Neuroticism (N)',legend=dict(orientation='h',y=-0.18))
fig.show()
print(f"canonical r(C,N) = {rcn:+.2f} over {len(cC)} chars -- correlation of rater-AVERAGED C and N.")
print("  (Estimand note: the 25-rater MEAN of each rater's OWN r(C,N) is -0.29; correlation-of-means (this cell) and mean-of-correlations are two valid views.)")
print(f"synth chars currently in the high-C/high-N corner: {in_hi} of 20 (none designed for it)")
print("The curation file includes a worked example (Edda Sarn) showing how to author a character into this empty corner.")

canonical r(C,N) = -0.38 over 58 chars -- correlation of rater-AVERAGED C and N.
  (Estimand note: the 25-rater MEAN of each rater's OWN r(C,N) is -0.29; correlation-of-means (this cell) and mean-of-correlations are two valid views.)
synth chars currently in the high-C/high-N corner: 0 of 20 (none designed for it)
The curation file includes a worked example (Edda Sarn) showing how to author a character into this empty corner.


## Per-character cards (designed targets + evidence-pack opening)

In [7]:
for c in chars:
    o=c['target_OCEAN']; h=c['target_HEXACO']
    ev=c['evidence_pack'] if isinstance(c['evidence_pack'],str) else ' '.join(c['evidence_pack'])
    print('='*78)
    print(f"{c['name']}   |   {c['work_title']}   |   {archetype(h['H'],h['A_HEX'])}")
    print(f"  OCEAN  O{o['O']:+.1f} C{o['C']:+.1f} E{o['E']:+.1f} A{o['A']:+.1f} N{o['N']:+.1f}")
    print(f"  HEXACO H{h['H']:+.2f} E{h['E']:+.2f} X{h['X']:+.2f} A_HEX{h['A_HEX']:+.2f} C{h['C']:+.2f} O{h['O']:+.2f}")
    print(f"  evidence: {ev[:240].strip()}...")


Dorrit Klaes   |   The Lichfield Dispatches   |   sincere-but-harsh (H+ A_HEX-)
  OCEAN  O+0.1 C+0.9 E-0.2 A-0.6 N-0.1
  HEXACO H+0.85 E-0.10 X-0.20 A_HEX-0.70 C+0.90 O+0.10
  evidence: "I'm going to tell you exactly what I think, and I don't care if it stings." Dorrit said this to the newly hired archivist on his first morning, arms crossed, standing in the doorway of the municipal records office. "Your filing proposal is...
Tessaro Vinn   |   A Gilded Residue   |   charming-but-crooked (H- A_HEX+)
  OCEAN  O+0.3 C+0.6 E+0.7 A+0.5 N-0.6
  HEXACO H-0.80 E-0.40 X+0.70 A_HEX+0.80 C+0.60 O+0.20
  evidence: "Oh, absolutely, take your time—I'm in no rush at all," Tessaro said, smiling warmly at the gallery owner who was fumbling with paperwork. He leaned against the counter with an easy posture, hands relaxed at his sides. "These things are alw...
Pella Sond   |   Undertow Season   |   aligned / other
  OCEAN  O+0.8 C-0.6 E+0.2 A+0.8 N+0.8
  HEXACO H+0.50 E+0.90 X+0.10 A_HEX+0.70 C-0.60 O+0

## Files created (the substrate and everything derived from it)

| File | What it holds |
|---|---|
| `pivot6_hexaco_atlas/synthetic_characters_substrate_v0.json` | the 20 characters: name, work, evidence pack, target OCEAN + HEXACO |
| the generation scripts in the umbrella repository | the generator (Claude Opus 4.6 + 3-LLM novelty control) |
| `pivot6_hexaco_atlas/ocean_ratings_*_synth.json`, `hexaco_ratings_*_synth.json` | the 26 rating slots scoring the synthetics (25 distinct raters after the Grok duplicate is dropped) |
| `pivot6_hexaco_atlas/synthetic_vs_canonical*.csv/.json` | the substrate-falsifier computation (Δ = -0.447 on the 25-rater panel) |
| `method_bakeoff_v4/synth_regressor_benchmark.json` | Mode 2: regressors on the synthetics (HEXACO head reproduces +0.84) |
| `pivot6_hexaco_atlas/synth_scpi_retrieval.json` | SCPI nearest canonical neighbors + predictions |
| `pivot6_hexaco_atlas/cache_membership_and_coverage.json` | the cache-membership gauge + coverage-hole stats |
| `pivot6_hexaco_atlas/cache_map_viz_data.json` | the NB05 projection data (t-SNE coords, cache scores) |

In [8]:
import os
A='paper_artifacts'
files=['pivot6_hexaco_atlas/synthetic_characters_substrate_v0.json',
       'method_bakeoff_v4/synth_regressor_benchmark.json',
       'pivot6_hexaco_atlas/synth_scpi_retrieval.json',
       'pivot6_hexaco_atlas/cache_membership_and_coverage.json',
       'pivot6_hexaco_atlas/cache_map_viz_data.json']
base=Path(A) if Path(A).exists() else Path('../'+A)
for f in files:
    fp=base/f; status=f'{fp.stat().st_size:,} bytes' if fp.exists() else 'NOT FOUND'
    print(f'{status:>16}  {f}')


    60,073 bytes  pivot6_hexaco_atlas/synthetic_characters_substrate_v0.json
    15,521 bytes  method_bakeoff_v4/synth_regressor_benchmark.json
    13,894 bytes  pivot6_hexaco_atlas/synth_scpi_retrieval.json
       805 bytes  pivot6_hexaco_atlas/cache_membership_and_coverage.json
   236,544 bytes  pivot6_hexaco_atlas/cache_map_viz_data.json


## Label propagation: the cache travels through training labels (paper §"Label propagation")

The substrate isn't only an inference-time probe. A regressor trained on **LLM-derived labels**
inherits the fusion even though it never saw the cache and never trained on these 20 characters.
The cell below reproduces `method_bakeoff_v4/synth_regressor_benchmark.json`: an OCEAN cheap regressor
recovers Openness/Conscientiousness/Extraversion but fails on Agreeableness and Neuroticism, and a
HEXACO head reproduces the H–A_HEX fusion at **r = +0.84** on characters designed at **−0.74**.

In [9]:
# --- Label propagation (Mode 2): does the cache travel through training labels? ---
# The regressors below never saw the cache and never trained on these 20 synthetic characters,
# yet a HEXACO head trained on LLM probe labels reproduces the H-A fusion on them.
bench = json.loads((base / 'method_bakeoff_v4' / 'synth_regressor_benchmark.json').read_text())
o = bench['ocean_per_trait_on_synth']
print('OCEAN cheap regressor recovery on the 20 synthetics (Ridge, trained on 562-char Consensus labels):')
for t in ['O', 'C', 'E', 'A', 'N']:
    r, p = o[t]['r'], o[t]['p']
    print(f"  {t}: r = {r:+.3f}  (p = {p:.3f})  -> {'recovers' if p < 0.05 else 'FAILS (n.s.)'}")
hb = bench['regressor_bipolarity_on_synth']['r_H_vs_A_HEX_predicted_on_synth']
des = bench['target_bipolarity_on_synth']['r_H_vs_A_HEX_author_targets']
print(f"\nHEXACO head, predicted H vs predicted A_HEX on the synthetics: r = {hb:+.3f}")
print(f"  designed (author targets):                                  r = {des:+.3f}")
print("\n=> The cheap regressor recovers O/C/E (r 0.63-0.80) but FAILS on the fused moral axis (A) and on N.")
print("   The HEXACO head reproduces the +0.84 fusion inherited from its LLM-derived labels,")
print("   sign-reversed from the designed -0.74: the cache propagates through training labels.")

OCEAN cheap regressor recovery on the 20 synthetics (Ridge, trained on 562-char Consensus labels):
  O: r = +0.797  (p = 0.000)  -> recovers
  C: r = +0.633  (p = 0.003)  -> recovers
  E: r = +0.672  (p = 0.001)  -> recovers
  A: r = +0.194  (p = 0.412)  -> FAILS (n.s.)
  N: r = -0.167  (p = 0.481)  -> FAILS (n.s.)

HEXACO head, predicted H vs predicted A_HEX on the synthetics: r = +0.837
  designed (author targets):                                  r = -0.744

=> The cheap regressor recovers O/C/E (r 0.63-0.80) but FAILS on the fused moral axis (A) and on N.
   The HEXACO head reproduces the +0.84 fusion inherited from its LLM-derived labels,
   sign-reversed from the designed -0.74: the cache propagates through training labels.


## Design your own synthetic character

Edit `my_character` below and run. It drops your character onto the trait-space
coverage map (canonical characters + the 20 synthetics), so you can see at a glance
whether it lands in a useful **decoupled corner** (a coverage hole, novel territory)
or on the **crowded fused diagonal** near canonical characters. Trait-space placement,
no embedding or API call needed; it answers "does my character fill a gap?" from the
H and A_HEX numbers alone.

(An embedding-space version, embed the character's text and place it among all ~2000
characters with a cache-membership score, is now possible: the canonical + synth
embeddings are persisted in `paper_artifacts/pivot6_hexaco_atlas/embeddings/`.)

In [10]:
# EDIT THIS: your candidate character's designed targets (scale -1.0 to +1.0)
my_character = {'name': 'My Character',
                'H': 0.85, 'A_HEX': -0.70,            # the two decoupled axes
                'O': 0.0, 'C': 0.5, 'E': 0.0, 'A': -0.5, 'N': 0.0}

vp = Path('paper_artifacts/pivot6_hexaco_atlas/cache_map_viz_data.json')
if not vp.exists(): vp = Path('../paper_artifacts/pivot6_hexaco_atlas/cache_map_viz_data.json')
V = json.loads(vp.read_text())
cH=[p['H'] for p in V['points'] if p['type']=='canonical' and p['H'] is not None]
cA=[p['A_HEX'] for p in V['points'] if p['type']=='canonical' and p['H'] is not None]
sH=[p['H'] for p in V['points'] if p['type']=='synth']; sA=[p['A_HEX'] for p in V['points'] if p['type']=='synth']
mh, ma = my_character['H'], my_character['A_HEX']

fig=go.Figure()
for (x0,x1,y0,y1) in [(0.3,1,-1,-0.3),(-1,-0.3,0.3,1)]:
    fig.add_shape(type='rect',x0=x0,x1=x1,y0=y0,y1=y1,fillcolor='#ffeaa7',opacity=0.45,line_width=0,layer='below')
fig.add_trace(go.Scatter(x=cH,y=cA,mode='markers',name='canonical',marker=dict(size=6,color='#b2bec3')))
fig.add_trace(go.Scatter(x=sH,y=sA,mode='markers',name='synthetic (20)',marker=dict(size=9,color='#D63031')))
fig.add_trace(go.Scatter(x=[mh],y=[ma],mode='markers+text',name=my_character['name'],
    marker=dict(size=22,color='#F9AB00',symbol='star',line=dict(width=1.5,color='#2d3436')),
    text=[my_character['name']],textposition='top center'))
fig.add_hline(y=0,line_color='#dfe6e9'); fig.add_vline(x=0,line_color='#dfe6e9')
fig.update_layout(template='plotly_white',height=560,xaxis_range=[-1,1],yaxis_range=[-1,1],
    title='Where your character lands (shaded = decoupled corners, the useful gaps)',
    xaxis_title='Honesty-Humility (H)',yaxis_title='HEXACO-Agreeableness (A_HEX)',legend=dict(orientation='h',y=-0.18))
fig.show()

if mh>0.3 and ma<-0.3: corner='sincere-but-harsh, a DECOUPLED corner (novel, useful)'
elif mh<-0.3 and ma>0.3: corner='charming-but-crooked, a DECOUPLED corner (novel, useful)'
else: corner='aligned / diagonal (crowded, close to canonical, less novel)'
dmin=min(((mh-h)**2+(ma-a)**2)**0.5 for h,a in zip(cH,cA))
print(f"{my_character['name']}: H={mh:+.2f}, A_HEX={ma:+.2f}")
print(f'  region: {corner}')
print(f'  nearest-canonical trait distance: {dmin:.2f}  (larger = deeper in a gap)')


My Character: H=+0.85, A_HEX=-0.70
  region: sincere-but-harsh, a DECOUPLED corner (novel, useful)
  nearest-canonical trait distance: 0.71  (larger = deeper in a gap)


### Embedding-space version: does my character's *text* land out-of-cache?

The trait-space check above plots the H/A_HEX numbers you *assert*. The real test embeds the character's
**text** and scores its cache-membership (max cosine similarity) against all 1981 canonical + 20 synthetic
characters, using the persisted 3072-d embeddings. A score below the canonical 5th percentile (~0.61) means
the text reads as novel territory, like the synthetics (median ~0.41). Needs one API call per character.

In [11]:
# Embedding-space placement: embed your character's TEXT, score cache-membership vs ~2000 chars
import os
my_name = 'My Character'
my_text = (
  "Paste a paragraph or two of your character's behavior and dialogue here. "
  "Concrete, observable detail places better than abstract description."
)
emb_dir = Path('paper_artifacts/pivot6_hexaco_atlas/embeddings')
if not emb_dir.exists(): emb_dir = Path('../paper_artifacts/pivot6_hexaco_atlas/embeddings')
zc=np.load(emb_dir/'canonical_embeddings_3072d.npz',allow_pickle=True)
zs=np.load(emb_dir/'synth_embeddings_3072d.npz',allow_pickle=True)
names=np.concatenate([zc['names'],zs['names']])
types=np.array(['canonical']*len(zc['names'])+['synth']*len(zs['names']))
M=np.vstack([zc['embeddings'].astype('float32'),zs['embeddings'].astype('float32')])
M/=np.linalg.norm(M,axis=1,keepdims=True)+1e-9

key=os.environ.get('OPENAI_API_KEY')
if not key:
    for envp in [Path('.env'),Path('../.env'),Path('.env')]:
        if envp.exists():
            for line in envp.read_text().splitlines():
                if line.startswith('OPENAI_API_KEY='): key=line.split('=',1)[1].strip().strip('"').strip("'"); break
        if key: break
if not key:
    print('No OPENAI_API_KEY found; set it (or set OPENAI_API_KEY) to run embedding-space placement.')
else:
    from openai import OpenAI
    v=np.array(OpenAI(api_key=key).embeddings.create(model='text-embedding-3-large',input=my_text).data[0].embedding,dtype='float32')
    v/=np.linalg.norm(v)+1e-9
    sims=M@v; gauge=float(sims.max()); order=np.argsort(-sims)[:6]
    verdict=('IN the cache (reads like a known character)' if gauge>0.61
             else 'OUT of cache (novel territory, like the synthetics)' if gauge<0.55
             else 'BORDERLINE (a bridge character)')
    print(f"{my_name}: cache-membership (max cosine) = {gauge:.3f}")
    print(f"  reference: canonical 5th pct ~ 0.61 | synth median ~ 0.41")
    print(f"  verdict: {verdict}\n  nearest neighbors:")
    for i in order: print(f"    {sims[i]:.3f}  [{types[i]:9s}] {names[i]}")

My Character: cache-membership (max cosine) = 0.390
  reference: canonical 5th pct ~ 0.61 | synth median ~ 0.41
  verdict: OUT of cache (novel territory, like the synthetics)
  nearest neighbors:
    0.390  [synth    ] Darvon Selk
    0.388  [synth    ] Quillen Dray
    0.387  [synth    ] Erith Cade Pollowick
    0.385  [synth    ] Tessaly Brune
    0.384  [canonical] Higgins
    0.383  [synth    ] Darveth Solke


## Reading

These 20 characters are the fulcrum of the paper: out-of-corpus by construction (and verified
so by a 3-LLM novelty control), with H and A_HEX deliberately decoupled (r = -0.74). Every
downstream finding, the substrate falsifier, the activation probe, Mode 2, the SCPI cache map,
is a measurement taken on this substrate. Companion notebooks: NB09 (the Catcher), NB05 (the
cache map).